In [1]:
from ml_enhance import QuantumFPDatasetBuilder, RDKitFeatureCalculator, get_preprocessed_smiles, canonicalize_smiles
from pathlib import Path
from rdkit import Chem
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def build_solubility_lookup(raw_df: pd.DataFrame) -> pd.Series:
    """
    Returns a Series indexed by canonical SMILES containing solubility.
    """
    df = raw_df[["SMILES", "Solubility"]].copy()

    df["canon_smiles"] = df["SMILES"].apply(canonicalize_smiles)
    df = df.dropna(subset=["canon_smiles"])

    return df.set_index("canon_smiles")["Solubility"]

def attach_solubility(feature_df: pd.DataFrame, sol_lookup: pd.Series) -> pd.DataFrame:
    df = feature_df.copy()

    df["canon_smiles"] = df["smiles"].apply(canonicalize_smiles)
    df["solubility"] = df["canon_smiles"].map(sol_lookup)

    return df

In [3]:
DATA_SOURCE = Path("../data/AqSolDB/data_curated.csv")
raw_data = pd.read_csv(DATA_SOURCE)
# QFP_preprocessed_smiles = np.array([get_preprocessed_smiles(smiles) for smiles in raw_data["SMILES"]])
# Write the smiles to JSON
# Run data through QuantumFP

In [4]:
QFP_OUTPUT_PATH = Path("../data/QuantumFP/QFP_output")
builder = QuantumFPDatasetBuilder(QFP_OUTPUT_PATH)
df, errors = builder.build_dataset_batched(batch_size=100)

100%|██████████| 89/89 [34:52<00:00, 23.51s/it]


In [5]:
errors

[]

In [6]:
rdkit_calculator = RDKitFeatureCalculator()
df = rdkit_calculator.add_to_dataframe(df, multiprocess=True)

100%|██████████| 8837/8837 [00:35<00:00, 247.84it/s]


In [7]:
# Add solutbility:
solubility_lookup = build_solubility_lookup(raw_data)
df = attach_solubility(df, solubility_lookup)

[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:55] WARNING: not removing hydrogen atom without neighbors
[10:31:56] WARNING: not removing hydrogen atom without neighbors
[10:31:56] WARNING: not removing hydrogen atom without neighbors
[10:31:56] WARNING: not removing hydrogen atom without neighbors
[10:31:56] WARNING: not r

In [8]:
df.head()

,id,atomization_energy,homo_lumo_gap,ionization_energy,electron_affinity,chemical_potential,molecular_dipole_norm,molecular_quadrupole_principal_invariant_2,molecular_quadrupole_principal_invariant_3,molecular_polarizability_mean,...,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,canon_smiles,solubility
0,0.0,5.239994,0.089513,0.476438,0.201907,-0.338539,1.513984,-191.687655,-37.755113,-99.744189,...,0,0,0,0,0,0,0,0,O=C1Nc2cccc3cccc1c23,-3.254767
1,1.0,3.330826,0.105979,0.507545,0.200001,-0.356143,0.965182,-188.636399,720.869344,-95.248037,...,0,0,0,0,0,0,0,0,O=Cc1ccc(Cl)cc1,-2.177078
2,10.0,7.140100,0.154440,0.463887,0.109782,-0.296101,0.909688,-169.958738,291.625442,-98.287723,...,0,0,0,0,0,0,0,0,CC(C)(C)c1ccc(OCC2CO2)cc1,-3.430239
3,100.0,5.850939,0.073945,0.299042,0.004866,-0.145684,5.103982,-1306.769545,16565.487506,-90.301117,...,0,0,0,0,0,0,0,0,Cc1cccc(C)c1OCC(=O)O,-2.256645
4,1000.0,8.519924,0.068689,0.479849,0.249515,-0.373551,2.145071,-670.362641,-1839.250752,-230.053983,...,0,0,0,0,0,0,0,0,COC(=O)c1ccc(C(=O)Nc2cc(Cl)ccc2Cl)cc1[N+](=O)[O-],-5.965155


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8837 entries, 0 to 8836
Columns: 354 entries, id to solubility
dtypes: float64(242), int64(110), str(2)
memory usage: 23.9 MB


In [10]:
df_clean = df.dropna() # mostly consisting of the molecules with transition metals as they cause problems with the partial charge calculation of RDKit
df_clean.info()

<class 'pandas.DataFrame'>
Index: 8763 entries, 0 to 8836
Columns: 354 entries, id to solubility
dtypes: float64(242), int64(110), str(2)
memory usage: 23.7 MB


In [11]:
# df_clean.to_csv("../data/processed_dataset_wo_metals_w_even_more_qm2.csv", index=False)

In [12]:
from collections import Counter

df_clean = df.dropna()

print()

total_atom_types = []
count = 0
for smiles in df_clean["canon_smiles"]:
    mol = Chem.MolFromSmiles(smiles)

    mol_atom_types = [atom.GetSymbol() for atom in mol.GetAtoms()]

    total_atom_types.extend(mol_atom_types)

print(Counter(total_atom_types))


Counter({'C': 104211, 'O': 21905, 'N': 10445, 'Cl': 3713, 'S': 1853, 'F': 1204, 'Br': 468, 'P': 332, 'I': 133, 'Si': 93, 'B': 7, 'Al': 1, 'H': 1})


In [13]:
Counter({'C': 104752, 'O': 22047, 'N': 10486, 'Cl': 3754, 'S': 1874, 'F': 1204, 'Br': 474, 'P': 332, 'I': 133, 'Si': 93, 'Sn': 18, 'Se': 10, 'B': 9, 'Pb': 6, 'Co': 5, 'Zn': 4, 'Cu': 4, 'As': 4, 'V': 3, 'Mo': 3, 'Cd': 3, 'Mn': 2, 'W': 2, 'Nb': 2, 'Cr': 2, 'Te': 2, 'Sb': 2, 'Ge': 2, 'Ca': 1, 'Bi': 1, 'Al': 1, 'Sr': 1, 'H': 1, 'Fe': 1, 'Ni': 1, 'Ti': 1, 'Zr': 1, 'Be': 1, 'Ce': 1, 'Hg': 1, 'Y': 1})

Counter({'C': 104752,
         'O': 22047,
         'N': 10486,
         'Cl': 3754,
         'S': 1874,
         'F': 1204,
         'Br': 474,
         'P': 332,
         'I': 133,
         'Si': 93,
         'Sn': 18,
         'Se': 10,
         'B': 9,
         'Pb': 6,
         'Co': 5,
         'Zn': 4,
         'Cu': 4,
         'As': 4,
         'V': 3,
         'Mo': 3,
         'Cd': 3,
         'Mn': 2,
         'W': 2,
         'Nb': 2,
         'Cr': 2,
         'Te': 2,
         'Sb': 2,
         'Ge': 2,
         'Ca': 1,
         'Bi': 1,
         'Al': 1,
         'Sr': 1,
         'H': 1,
         'Fe': 1,
         'Ni': 1,
         'Ti': 1,
         'Zr': 1,
         'Be': 1,
         'Ce': 1,
         'Hg': 1,
         'Y': 1})